# ROIs → reference brain, one experiment

Puts every suite2p ROI of each fish of one experiment into the common reference brain (`ref_L433_f02`), using the
anatomy → reference registration that is already done. Each fish gets one table, written next to its other
reference-registration outputs and never overwritten:
`<fish>/02_reg/08_2pa-ref/roi_positions/ref_L433_f02/<fish_id>_roi_positions_in_ref.csv`.
QC figures and a QC table go to `Paper/2p_derived/registration/<experiment>/rois_in_ref/`.
Rerun it for a new experiment by changing `EXPERIMENT`; fish already done are skipped.

## 1. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Walk up to the repo root (marked by utils.py) so the project modules import from anywhere.
repo_root = Path.cwd()
while not (repo_root / "utils.py").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from registration.batch_roi_to_reference import (
    STATUS_DONE,
    STATUS_READY,
    check_plane_depth_order,
    map_fish_rois_to_reference,
    read_warped_anatomy,
    reference_mapping_status,
    reference_roi_table_path,
    score_fish_roi_mapping,
    write_reference_roi_table,
)
from registration.experiment_fish import find_experiment_fish_folders, list_experiment_fish
from registration.image_utils import norm01
from registration.qc_plotting import save_figure
from registration.reference_qc import (
    add_decision_columns,
    read_nrrd_stack,
    read_previous_decisions,
    reference_anatomy_path,
    scale_fish_image,
    write_qc_table,
)
from registration.roi_to_reference import read_nrrd_geometry

## 2. Settings
Choose the experiment here.

In [ ]:
EXPERIMENT = "Exp_8_accumulation_ev"  # Exp_1_flickering, Exp_5_map_positions, Exp_7_trans_stat, Exp_8_accumulation_ev, Exp_9_RF_memory_inhibition

LAB_ROOT = Path(r"C:\Users\suribear\OneDrive - Université de Lausanne\Lab")
DATA_ROOT = LAB_ROOT / "Data" / "2p"  # one folder per fish, named "<fish_id>..."
OVERVIEW_CSV = LAB_ROOT / "Results" / "Paper" / "2p_derived" / "global" / "overview.csv"
QC_DIR = LAB_ROOT / "Results" / "Paper" / "2p_derived" / "registration" / EXPERIMENT / "rois_in_ref"
QC_TABLE_PATH = QC_DIR / f"{EXPERIMENT}_rois_in_ref_qc.csv"
OVERVIEW_PATH = QC_DIR / f"{EXPERIMENT}_rois_in_ref_overview.png"

REFERENCE_NAME = "ref_L433_f02"
REFERENCE_FISH_ID = "L433_f02"
REFERENCE_FOLDER = DATA_ROOT / "L433_f02_Exp_1_flickering"

MIN_AGREEMENT = 0.9  # ROIs should hit the same anatomy in the warped fish; pilot L800_f04 gave r = 0.996
MIN_FRACTION_INSIDE = 0.95  # share of ROIs that must land inside the warped fish's data

## 3. Reference brain
Loaded once; its grid converts micrometres to reference voxels.

In [ ]:
reference_path = reference_anatomy_path(REFERENCE_FOLDER)
reference_geometry = read_nrrd_geometry(reference_path)
reference_stack = read_nrrd_stack(reference_path)
print(f"Reference: {reference_stack.shape} (z, y, x), voxel {reference_geometry['spacing_xyz_um']} µm (x, y, z)")

## 4. Which fish can be mapped
The experiment's fish from the Paper overview (without `exclude`). `ready` fish are mapped below, `done` fish are skipped.

In [ ]:
experiment_fish = list_experiment_fish(OVERVIEW_CSV, EXPERIMENT)
fish_folders = find_experiment_fish_folders(DATA_ROOT, experiment_fish)
status_table = pd.DataFrame([
    {"fish_id": fish["fish_id"], "overview_decision": fish["final_decision"],
     "status": reference_mapping_status(fish_folders[fish["fish_id"]], fish["fish_id"], REFERENCE_NAME, REFERENCE_FISH_ID)}
    for fish in experiment_fish
])
status_table

## 5. Map the ROIs of every ready fish
One fish at a time; a fish that fails is reported and skipped. Each warp file is ~1 GB, so this takes seconds to minutes per fish.

In [ ]:
mapped_fish, skipped_fish = [], {}
for fish in experiment_fish:
    fish_id, fish_folder = fish["fish_id"], fish_folders[fish["fish_id"]]
    status = reference_mapping_status(fish_folder, fish_id, REFERENCE_NAME, REFERENCE_FISH_ID)
    if status != STATUS_READY:
        if status != STATUS_DONE:
            skipped_fish[fish_id] = status
        continue
    print(f"=== {fish_id}")
    try:
        reference_table, provenance = map_fish_rois_to_reference(fish_folder, fish_id, reference_geometry, REFERENCE_NAME, REFERENCE_FISH_ID)
        write_reference_roi_table(reference_table, reference_roi_table_path(fish_folder, fish_id, REFERENCE_NAME), {**provenance, "experiment": EXPERIMENT})
    except (OSError, RuntimeError, ValueError) as error:
        skipped_fish[fish_id] = f"could not map: {error}"
        continue
    mapped_fish.append(fish_id)

for fish_id, reason in skipped_fish.items():
    print(f"WARNING {fish_id}: skipped -- {reason}")
print(f"Mapped {len(mapped_fish)} fish now, skipped {len(skipped_fish)}.")

## 6. QC figure layout
Top and side views of the fish's ROIs on the reference, then one panel per imaging plane on the fish's own warped anatomy. Edit sizes and colours here.

In [ ]:
POSITION_COLUMNS = ["x_ref_vox", "y_ref_vox", "z_ref_vox"]
PLANE_COLORS = plt.get_cmap("tab10")
REFERENCE_CLIP_PERCENTILES = (1, 99.5)
FISH_CLIP_PERCENTILES = (10, 99.5)
DEPTH_HALF_WIDTH_SLICES = 1  # ROIs within +-1 reference slice of a plane's depth are drawn on it
CROP_MARGIN_PX = 20  # margin around the ROIs, reference pixels
DOT_SIZE = 3
PANEL_SIZE = 3.5  # inches
QC_DPI = 100

# Reference views, shared by every fish: top = max over depth, side = max over y (depth vs x).
reference_top = norm01(reference_stack.max(axis=0), *REFERENCE_CLIP_PERCENTILES)
reference_side = norm01(reference_stack.max(axis=1), *REFERENCE_CLIP_PERCENTILES)
SIDE_ASPECT = reference_geometry["spacing_xyz_um"][2] / reference_geometry["spacing_xyz_um"][0]  # true proportions


def plane_depths(reference_table):
    """Median reference depth (slice) of each imaging plane's ROIs.

    Args:
        reference_table (pandas.DataFrame): One fish's ROI table in reference space.

    Returns:
        dict: plane -> rounded reference slice index.
    """
    medians = reference_table.groupby("plane")["z_ref_vox"].median().dropna()
    depths = {int(plane): int(round(depth)) for plane, depth in medians.items()}
    return depths


def draw_views(top_axis, side_axis, positions, colors, size=DOT_SIZE, alpha=0.8):
    """Draw ROI centres on the reference top and side views.

    Args:
        top_axis (matplotlib.axes.Axes): Axis for the top view (x, y).
        side_axis (matplotlib.axes.Axes): Axis for the side view (x, depth).
        positions (numpy.ndarray): (n, 3) reference voxels, x, y, z.
        colors: One colour or (n,) colours.
        size (float): Dot size.
        alpha (float): Dot opacity.
    """
    top_axis.imshow(reference_top, cmap="gray")
    top_axis.scatter(positions[:, 0], positions[:, 1], s=size, c=colors, alpha=alpha, linewidths=0)
    side_axis.imshow(reference_side, cmap="gray", aspect=SIDE_ASPECT)
    side_axis.scatter(positions[:, 0], positions[:, 2], s=size, c=colors, alpha=alpha, linewidths=0)
    top_axis.set_title("top view (reference)")
    side_axis.set_title("side view (reference)")
    for axis in (top_axis, side_axis):
        axis.set_xticks([])
        axis.set_yticks([])


def plot_fish_rois(fish_id, reference_table, warped_stack, score):
    """One fish's mapped ROIs: reference top/side views, then each plane on its warped anatomy.

    Args:
        fish_id (str): Fish identifier.
        reference_table (pandas.DataFrame): The fish's ROI table in reference space.
        warped_stack (numpy.ndarray): The fish's anatomy on the reference grid.
        score (dict): From `score_fish_roi_mapping`.

    Returns:
        matplotlib.figure.Figure: The figure.
    """
    positioned = reference_table.dropna(subset=POSITION_COLUMNS)
    positions = positioned[POSITION_COLUMNS].to_numpy()
    colors = [PLANE_COLORS(int(plane) % 10) for plane in positioned["plane"]]
    depths = plane_depths(positioned)
    n_panels = 2 + len(depths)
    figure, axes = plt.subplots(1, n_panels, figsize=(n_panels * PANEL_SIZE, PANEL_SIZE), squeeze=False, layout="compressed")
    draw_views(axes[0, 0], axes[0, 1], positions, colors)
    x0, y0 = np.maximum(positions[:, :2].min(axis=0) - CROP_MARGIN_PX, 0).astype(int)
    x1, y1 = (positions[:, :2].max(axis=0) + CROP_MARGIN_PX).astype(int)
    for axis, (plane, depth) in zip(axes[0, 2:], depths.items()):
        near_depth = np.abs(positions[:, 2] - depth) <= DEPTH_HALF_WIDTH_SLICES
        axis.imshow(scale_fish_image(warped_stack[depth, y0:y1, x0:x1], *FISH_CLIP_PERCENTILES), cmap="gray")
        axis.scatter(positions[near_depth, 0] - x0, positions[near_depth, 1] - y0, s=DOT_SIZE,
                     c=[colors[index] for index in np.flatnonzero(near_depth)], linewidths=0)
        axis.set_title(f"plane {plane}: z={depth}, {near_depth.sum()} ROIs within ±{DEPTH_HALF_WIDTH_SLICES}")
        axis.set_xticks([])
        axis.set_yticks([])
    figure.suptitle(f"{fish_id} → {REFERENCE_NAME}: ROIs hit the same anatomy r = {score['correlation']:.3f}, "
                    f"{score['fraction_inside']:.0%} inside, {score['n_rois_without_position']} without position (colour = plane)")
    return figure

## 7. Score and plot each mapped fish
Checks that every fish's ROIs land on the same anatomy in its warped image, and saves one figure per fish.

In [ ]:
summary_rows, fish_positions = [], {}
for fish in experiment_fish:
    fish_id, fish_folder = fish["fish_id"], fish_folders[fish["fish_id"]]
    if reference_mapping_status(fish_folder, fish_id, REFERENCE_NAME, REFERENCE_FISH_ID) != STATUS_DONE:
        continue
    reference_table = pd.read_csv(reference_roi_table_path(fish_folder, fish_id, REFERENCE_NAME))
    warped_stack = read_warped_anatomy(fish_folder, fish_id, REFERENCE_NAME, reference_stack, REFERENCE_FISH_ID)
    score = score_fish_roi_mapping(fish_folder, reference_table, warped_stack, reference_stack)
    save_figure(plot_fish_rois(fish_id, reference_table, warped_stack, score), QC_DIR / f"{fish_id}_rois_in_ref.png", QC_DPI)
    del warped_stack
    gc.collect()
    cells_only = reference_table.loc[reference_table["is_cell"].astype(bool)].dropna(subset=POSITION_COLUMNS)
    fish_positions[fish_id] = cells_only[POSITION_COLUMNS].to_numpy()
    depth_order = check_plane_depth_order(reference_table)
    summary_rows.append({"fish_id": fish_id, "overview_decision": fish["final_decision"], **score, **depth_order})
    print(f"{fish_id}: r = {score['correlation']:.3f}, inside {score['fraction_inside']:.1%}, plane depths {depth_order['plane_depths']}")

## 8. QC table
Lowest agreement first. `check` marks fish under the thresholds, or whose imaging planes share an anatomy depth or are out of order (a step 1 plane-matching problem that step 3 cannot fix). Write `ok` / `check` / `bad` and a comment in the CSV's `decision` and `comment` columns: they are kept when this notebook is rerun.

In [ ]:
qc_table = pd.DataFrame(summary_rows).sort_values("correlation", ignore_index=True)
qc_table["check"] = ((qc_table["correlation"] < MIN_AGREEMENT) | (qc_table["fraction_inside"] < MIN_FRACTION_INSIDE)
                     | qc_table["duplicate_plane_depths"] | qc_table["plane_depths_out_of_order"])
qc_table = add_decision_columns(qc_table, read_previous_decisions(QC_TABLE_PATH))
write_qc_table(qc_table, QC_TABLE_PATH, {
    "experiment": EXPERIMENT,
    "reference_name": REFERENCE_NAME,
    "reference_path": reference_path,
    "min_agreement": MIN_AGREEMENT,
    "min_fraction_inside": MIN_FRACTION_INSIDE,
    "skipped_fish": skipped_fish,
})
qc_table[["fish_id", "correlation", "fraction_inside", "reference_correlation", "n_rois", "n_rois_without_position",
          "plane_depths", "check", "overview_decision", "decision", "comment"]]

## 9. All fish of the experiment
Suite2p cells of every fish together in the reference (one colour per fish). Fish should overlap in the same brain regions.

In [ ]:
FISH_COLORS = plt.get_cmap("tab20")
LEGEND_MARKER_SIZE = 4
figure, axes = plt.subplots(1, 2, figsize=(2 * PANEL_SIZE * 1.6, PANEL_SIZE * 1.6), layout="compressed")
legend_handles = []
for fish_index, (fish_id, positions) in enumerate(fish_positions.items()):
    draw_views(axes[0], axes[1], positions, [FISH_COLORS(fish_index % 20)], size=1, alpha=0.5)
    legend_handles.append(plt.Line2D([], [], linestyle="", marker="o", markersize=LEGEND_MARKER_SIZE, color=FISH_COLORS(fish_index % 20), label=fish_id))
figure.legend(handles=legend_handles, loc="outside right center", fontsize=7, frameon=False)
figure.suptitle(f"{EXPERIMENT}: suite2p cells of {len(fish_positions)} fish in {REFERENCE_NAME}")
save_figure(figure, OVERVIEW_PATH, QC_DPI)
print(f"Saved to {QC_DIR}")